In [1]:
import os
from dotenv import load_dotenv

import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

load_dotenv()

True

In [2]:
cimp = pd.read_csv('results_557k/cluster_sufficiency_classification_2026-03-19.csv')
cimp.head()

,cluster_id,text,count,context,stage1_reasoning,stage1_classification,stage2_reasoning,stage2_classification
0,0,Insurance programs and tax incentives for floo...,715,Cluster 0 - Representative policy:\n\nInsuranc...,The policy of insurance programs and tax incen...,SUFFICIENCY-COMPATIBLE,The policy is classified as Potential Sufficie...,PS
1,1,Preserving natural lands for flood management,447,Cluster 1 - Representative policy:\n\nPreservi...,The policy 'Preserving natural lands for flood...,SUFFICIENCY-COMPATIBLE,The policy is classified as 'Sufficiency (S)' ...,S
2,2,Continuous monitoring of fishways to optimise ...,1536,Cluster 2 - Representative policy:\n\nContinuo...,The policy involves continuous monitoring of f...,SUFFICIENCY-COMPATIBLE,The policy is classified as Sufficiency (S) be...,S
3,3,Increasing restoration funding,79,Cluster 3 - Representative policy:\n\nIncreasi...,The policy of increasing restoration funding i...,SUFFICIENCY-COMPATIBLE,The policy of increasing restoration funding p...,S
4,4,Implementation of measures to mitigate the ris...,1031,Cluster 4 - Representative policy:\n\nImplemen...,The policy focuses on mitigating flood risks i...,SUFFICIENCY-COMPATIBLE,The policy primarily and directly contributes ...,S


In [3]:
cemb = pd.read_parquet('data/clusters_representatives_with_embeddings_Qwen3-8B.parquet')
cemb.head()

,cluster_id,text,count,embedding
0,0,Insurance programs and tax incentives for floo...,715,"[0.013568851165473461, 0.02100222185254097, -0..."
1,1,Preserving natural lands for flood management,447,"[0.02139049768447876, -0.0014260332100093365, ..."
2,2,Continuous monitoring of fishways to optimise ...,1536,"[-0.0037227999418973923, -0.009593934752047062..."
3,3,Increasing restoration funding,79,"[-0.011206850409507751, 0.02045968547463417, -..."
4,4,Implementation of measures to mitigate the ris...,1031,"[0.009541078470647335, 0.029512155801057816, -..."


In [4]:
all(cimp.cluster_id == cemb.cluster_id)

True

In [5]:
df = cimp.merge(cemb[['cluster_id', 'embedding']], on='cluster_id')
df.head()

,cluster_id,text,count,context,stage1_reasoning,stage1_classification,stage2_reasoning,stage2_classification,embedding
0,0,Insurance programs and tax incentives for floo...,715,Cluster 0 - Representative policy:\n\nInsuranc...,The policy of insurance programs and tax incen...,SUFFICIENCY-COMPATIBLE,The policy is classified as Potential Sufficie...,PS,"[0.013568851165473461, 0.02100222185254097, -0..."
1,1,Preserving natural lands for flood management,447,Cluster 1 - Representative policy:\n\nPreservi...,The policy 'Preserving natural lands for flood...,SUFFICIENCY-COMPATIBLE,The policy is classified as 'Sufficiency (S)' ...,S,"[0.02139049768447876, -0.0014260332100093365, ..."
2,2,Continuous monitoring of fishways to optimise ...,1536,Cluster 2 - Representative policy:\n\nContinuo...,The policy involves continuous monitoring of f...,SUFFICIENCY-COMPATIBLE,The policy is classified as Sufficiency (S) be...,S,"[-0.0037227999418973923, -0.009593934752047062..."
3,3,Increasing restoration funding,79,Cluster 3 - Representative policy:\n\nIncreasi...,The policy of increasing restoration funding i...,SUFFICIENCY-COMPATIBLE,The policy of increasing restoration funding p...,S,"[-0.011206850409507751, 0.02045968547463417, -..."
4,4,Implementation of measures to mitigate the ris...,1031,Cluster 4 - Representative policy:\n\nImplemen...,The policy focuses on mitigating flood risks i...,SUFFICIENCY-COMPATIBLE,The policy primarily and directly contributes ...,S,"[0.009541078470647335, 0.029512155801057816, -..."


In [6]:
df.stage1_classification.value_counts()

stage1_classification
SUFFICIENCY-COMPATIBLE    1848
EFFICIENCY                 280
NOT-COMPATIBLE             218
DECARBONATION              135
Name: count, dtype: int64

In [7]:
df.stage2_classification.value_counts()

stage2_classification
S     1327
NS     582
PS     522
Name: count, dtype: int64

In [8]:
sdf = df[(df.stage1_classification == 'SUFFICIENCY-COMPATIBLE') & (df.stage2_classification.isin(['S', 'PS']))]
print(len(df), '-->', len(sdf))

2625 --> 1848


In [9]:
imp = pd.read_parquet('results_557k/cluster_impacts_2026-03-12.parquet')
imp.head()

,cluster_id,impact_category,impact_dim,negative,neutral,positive,negative_refs,neutral_refs,positive_refs
0,0,Human Needs,Shelter and living conditions,0,0,13,[],[],"[W2003373721_1, W2156345725_0, W2289768102_1, ..."
1,0,Justice,Distributional,0,2,0,[],"[W2557096622_1, W4389432677_0]",[]
2,0,Planetary Boundaries,Climate change,0,0,1,[],[],[W4415811729_2]
3,0,Wellbeing,Community,0,0,1,[],[],[W4287095827_3]
4,0,Wellbeing,Environment,0,0,8,[],[],"[W1000416519_1, W2156345725_0, W2557096622_1, ..."


In [10]:
d = {}
for (cid, cat, impact), group in imp.groupby(['cluster_id', 'impact_category', 'impact_dim']):
    tmp = group.drop(['cluster_id', 'impact_category', 'impact_dim'], axis=1)
    tmp = tmp.map(lambda x: x.tolist() if hasattr(x, 'tolist') else x)
    payload = tmp.to_dict(orient='records')
    if cid not in d:
        d[cid] = {cat: {impact: payload}}
    else:
        if cat not in d[cid]:
            d[cid][cat] = {impact: payload}
        else:
            d[cid][cat][impact] = payload

In [11]:
sdf['impacts'] = sdf['cluster_id'].map(lambda x: d.get(x, {}))

In [12]:
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
)
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='clusters-v1'), CollectionDescription(name='library-v1'), CollectionDescription(name='policies-v1')])

In [13]:
collection = "clusters-v260319"
dim = 4096

In [14]:
client.create_collection(
    collection_name=collection,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
)

True

In [15]:
vectors = sdf.pop('embedding')
payloads = [row.to_dict() for _, row in sdf.iterrows()]

In [16]:
payloads[0]

{'cluster_id': 0,
 'text': 'Insurance programs and tax incentives for flood risk',
 'count': 715,
 'context': "Cluster 0 - Representative policy:\n\nInsurance programs and tax incentives for flood risk\n\nImpacts assessment summary :\n\nHuman Needs\n- Shelter and living conditions\n  - positive: 13\n  - negative: 0\n  - neutral: 0\n\nJustice\n- Distributional\n  - positive: 0\n  - negative: 0\n  - neutral: 2\n\nPlanetary Boundaries\n- Climate change\n  - positive: 1\n  - negative: 0\n  - neutral: 0\n\nWellbeing\n- Community\n  - positive: 1\n  - negative: 0\n  - neutral: 0\n\n\n- Environment\n  - positive: 8\n  - negative: 0\n  - neutral: 0\n\n\n- Housing\n  - positive: 7\n  - negative: 0\n  - neutral: 0\n\n\n- Income\n  - positive: 2\n  - negative: 0\n  - neutral: 0\n\n\n- Safety\n  - positive: 12\n  - negative: 0\n  - neutral: 0\n\nImpacts assessment context samples:\n\nHuman Needs - Shelter and living conditions:\n\nPOSITIVE example:\n>  a study [36], this was found to reduce conten

In [17]:
client.upload_collection(
    collection,
    vectors=vectors,
    payload=payloads,
    parallel=4,
    batch_size=100
)

In [18]:
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='clusters-v1'), CollectionDescription(name='clusters-v260319'), CollectionDescription(name='library-v1'), CollectionDescription(name='policies-v1')])